# Cortical TRF analysis 3: does SI-GEVD denoising actually help?

Four sub-questions, each comparing the plain `raw='ica'` pipeline against
`raw='ica-sigevd'` (see `experiment_sigevd.py`), across every condition
with real two-talker or single-talker EEG - clean speech, diotic,
dichotic-left, dichotic-right, binaural - and both the onset
(`gammatone-on-8`) and envelope (`gammatone-8`) predictors:

1. **TRF estimation accuracy** - relative MSE between each subject's own
   estimated TRF and a group-average "ground truth" TRF (computed from the
   *raw*, non-denoised pipeline, so neither method is compared against a
   benchmark derived from itself).
2. **Neural tracking (Pearson r)** - each pipeline's own unique
   contribution (`compute_unique_contribution`, reused unmodified from
   `cortical_analysis_2.ipynb`) for that predictor, then raw vs SI-GEVD
   directly.
3. **Neural tracking (proportion of variance explained)** - identical to
   (2), `metric='ev'`.
4. **Attention decoding accuracy** (two-talker conditions only - clean
   speech has nothing to decode) - a Biesmans et al. 2017-style
   stimulus-reconstruction decoder: fit one TRF per held-out trial on the
   attended stream, apply it to both streams, classify attended as
   whichever correlates better with the real EEG, swept across several
   trial lengths.

**Every statistical test here is a paired Wilcoxon signed-rank test**
(`sigevd_eval.paired_wilcoxon`/`per_sensor_wilcoxon_mask`), not eelbrain's
own cluster-permutation test used elsewhere in this project - a single
`e.load_model_test()` call can only test within one raw pipeline at a
time, so there's no eelbrain result object spanning both `'ica'` and
`'ica-sigevd'` to draw a cluster-corrected mask from. Threshold: p < 0.025
throughout (Bonferroni correction for 2 comparisons, as specified).

**This notebook has never been run.** `sigevd_eval.py`'s numeric functions
are validated against synthetic data (see that module's own docstrings),
but nothing here has been exercised against eelbrain or this project's
real dataset - expect live-debugging rounds, same as
`cortical_analysis_2.ipynb`'s own SI-GEVD integration. The attention-
decoding section in particular (sub-question 4) is a substantial new
computation - leave-one-trial-out TRF refitting per subject, per
condition, per predictor, per pipeline - and is likely to take a long
time to run in full.

In [ ]:
from eelbrain import *
from experiment import topomap_with_colorbar, FIGURES_DIR
from experiment_sigevd import e as e_sigevd, load_decoding_trials
from experiment import e as e_raw
from sigevd_eval import relative_mse, paired_wilcoxon, per_sensor_wilcoxon_mask, decode_attention_accuracy
import matplotlib.pyplot as plt
import numpy
import seaborn as sns

# Where to save plots - inside the BIDS dataset's derivatives, same as
# cortical_analysis.ipynb/cortical_analysis_2.ipynb, not somewhere outside
# the dataset.
DST = FIGURES_DIR

## Preprocessing (run once per subject)

Identical to `cortical_analysis_2.ipynb` - both notebooks use the same
preprocessed EEG (`raw='ica'`), so this only does real work the first
time any of the three notebooks is run for a given subject. `'ica-sigevd'`
builds on top of `'ica'` (see `experiment_sigevd.py`) rather than
needing its own separate bad-channels/ICA step, so this only needs to
run against `e_raw`.

In [ ]:
MANUAL_BAD_CHANNELS = False
AUTO_BAD_CHANNELS_R = 0.3
REDO_BAD_CHANNELS = True

e_raw.mark_bad_channels(
    'all',
    manual_bad_channels=MANUAL_BAD_CHANNELS, auto_bad_channels_r=AUTO_BAD_CHANNELS_R,
    redo_bad_channels=REDO_BAD_CHANNELS,
)

In [ ]:
MANUAL_ICA = False
AUTO_ICA_CONFIDENCE = 0.75

e_raw.select_ica_artifacts(
    'all',
    manual_ica=MANUAL_ICA, auto_ica_confidence=AUTO_ICA_CONFIDENCE,
)

## Shared setup

Everything in this section is reused by every sub-question/condition
below. Mostly identical to `cortical_analysis_2.ipynb`'s own shared setup
(reused verbatim where possible), plus the raw-vs-SI-GEVD comparison
infrastructure this notebook adds.

In [ ]:
ATTEND, IGNORE, MIX = '', 'bg~', 'mix~'


def envelope(role=ATTEND):
    return f"{role}gammatone-8"


def onset(role=ATTEND):
    return f"{role}gammatone-on-8"


FULL_CLEAN = f"{envelope(ATTEND)} + {onset(ATTEND)}"
FULL_BASE = f"{envelope(ATTEND)} + {onset(ATTEND)} + {envelope(IGNORE)} + {onset(IGNORE)} + {envelope(MIX)} + {onset(MIX)}"

In [ ]:
METRIC = 'r'  # overridden per call below where sub-question 3 needs 'ev' instead
PMIN = 0.05
TOPO_ARGS = dict(clip='circle')

### Two pipelines, side by side

Unlike `cortical_analysis_2.ipynb` (which swaps the single global `e`/
`PARAMETERS` in place), this notebook needs *both* pipelines available at
once to compare them - `e_raw`/`PARAMETERS_RAW` (plain `'ica'`) and
`e_sigevd`/`PARAMETERS_SIGEVD` (`'ica-sigevd'`), never reassigned.

In [ ]:
PARAMETERS_RAW = {
    'raw': 'ica',
    'group': 'all',
    'samplingrate': 128,
    'data': 'eeg',
    'tstart': -0.100,
    'tstop': 0.600,
}
PARAMETERS_SIGEVD = {**PARAMETERS_RAW, 'raw': 'ica-sigevd'}

LAG_START_MS = PARAMETERS_RAW['tstart'] * 1e3
LAG_STOP_MS = PARAMETERS_RAW['tstop'] * 1e3
LAM = 0.2  # matches experiment_sigevd.py's own SIGEVD_LAM - the paper's example value, not tuned for this dataset

data = e_raw.load_trfs(1, 'gammatone-8', epoch='clean', **PARAMETERS_RAW)
eeg = data['ev']

ROI = [
    'AF3', 'F1', 'F3', 'F5',
    'FC5', 'FC3', 'FC1',
    'C1', 'C3', 'C5',
    'AF4', 'AFz',
    'Fz', 'F2', 'F4', 'F6',
    'FC6', 'FC4', 'FC2', 'FCz',
    'Cz', 'C2', 'C4', 'C6',
]

p = plot.SensorMap(eeg, mark=ROI)

### Model-comparison helpers

`compute_unique_contribution` is reused unmodified from
`cortical_analysis_2.ipynb` - it reads `e`/`PARAMETERS` from this
module's global scope, so each sub-question 2/3 cell below temporarily
points them at one pipeline, calls it, then the other, restoring the
previous values afterward (never left swapped between cells, unlike
`cortical_analysis_2.ipynb`, since this notebook needs both pipelines
throughout).

In [ ]:
def compute_unique_contribution(full_model, predictor, epoch, roi=ROI, metric=METRIC):
    "Omission comparison: does `predictor` uniquely improve on `full_model` with it removed?"
    comparison = f"{full_model} @ {predictor}"
    data, result = e.load_model_test(comparison, **PARAMETERS, pmin=PMIN, metric=metric, epoch=epoch, return_data=True)
    diff = table.difference(metric, 'model', 'test', 'baseline', 'subject', data=data)
    diff[f'roi_{metric}'] = diff[metric].mean(sensor=roi)
    return data, diff, result


def run_unique_contribution_both_pipelines(full_model, predictor, epoch, metric=METRIC):
    "compute_unique_contribution() under both e_raw/PARAMETERS_RAW and e_sigevd/PARAMETERS_SIGEVD - nothing else in this notebook reads bare e/PARAMETERS, so there's nothing to restore afterward"
    global e, PARAMETERS
    e, PARAMETERS = e_raw, PARAMETERS_RAW
    _, diff_raw, _ = compute_unique_contribution(full_model, predictor, epoch, metric=metric)
    e, PARAMETERS = e_sigevd, PARAMETERS_SIGEVD
    _, diff_sigevd, _ = compute_unique_contribution(full_model, predictor, epoch, metric=metric)
    return diff_raw, diff_sigevd

### TRF-reading helpers

`extract_trf` is reused unmodified from `cortical_analysis_2.ipynb`.

In [ ]:
def extract_trf(data_full, column, roi=ROI):
    "Per-subject, ROI-averaged, frequency-summed, smoothed TRF for one predictor column, read from an already-loaded load_trfs() result"
    trf = data_full[column].sum('frequency').mean(sensor=roi)
    ds = data_full['subject',]
    ds['trf_sm'] = resample(trf, 512)
    return ds

### Sub-question 1: TRF estimation accuracy (relative MSE)

Ground truth = the group-average (across all subjects) *raw*-pipeline TRF
for that predictor/condition - not SI-GEVD's own, so SI-GEVD isn't
compared against a benchmark derived from itself. Each subject's own TRF
(both pipelines) is scored by relative MSE against that one ground truth.

In [ ]:
PIPELINE_COLORS = {'Raw': '#6aa84f', 'SI-GEVD': '#3d85c6'}


def plot_bar_swarm_comparison(values_raw, values_sigevd, ylabel, title):
    "Side-by-side comparison, raw vs SI-GEVD: a bar plot (mean +/- SEM) and a per-subject swarm plot, sharing one y-axis label"
    labels = ['Raw'] * len(values_raw) + ['SI-GEVD'] * len(values_sigevd)
    values = numpy.concatenate([values_raw, values_sigevd])

    fig, axes = plt.subplots(1, 2, figsize=(6, 3.5))
    sns.barplot(x=labels, y=values, hue=labels, palette=PIPELINE_COLORS, legend=False, errorbar='se', ax=axes[0])
    axes[0].set_ylabel(ylabel)
    sns.swarmplot(x=labels, y=values, hue=labels, palette=PIPELINE_COLORS, legend=False, size=5, ax=axes[1])
    axes[1].set_ylabel(ylabel)
    fig.suptitle(title, fontsize=9)
    fig.tight_layout()
    return fig


def section_trf_accuracy(data_full_raw, data_full_sigevd, column, label, epoch):
    trf_raw = extract_trf(data_full_raw, column)
    trf_sigevd = extract_trf(data_full_sigevd, column)
    ground_truth = trf_raw['trf_sm'].mean('case').get_data(('time',))

    # trf_raw/trf_sigevd are Datasets - a dict subclass, so len() gives the
    # number of *columns* (2: 'subject', 'trf_sm'), not the number of
    # subjects. n_cases is the actual per-subject row count.
    mse_raw = numpy.array([relative_mse(trf_raw['trf_sm'][i].get_data(('time',)), ground_truth) for i in range(trf_raw.n_cases)])
    mse_sigevd = numpy.array([relative_mse(trf_sigevd['trf_sm'][i].get_data(('time',)), ground_truth) for i in range(trf_sigevd.n_cases)])

    stat, p, sig = paired_wilcoxon(mse_raw, mse_sigevd)
    print(f"{label} TRF accuracy ({epoch}): raw mean={mse_raw.mean():.4g}, SI-GEVD mean={mse_sigevd.mean():.4g}, Wilcoxon W={stat:.3f}, p={p:.4f}, significant (p<0.025)={sig}")

    return plot_bar_swarm_comparison(mse_raw, mse_sigevd, 'Relative MSE', f'{label} TRF accuracy ({epoch})')

### Sub-questions 2/3: neural tracking (r / ev), raw vs SI-GEVD

Each pipeline's own unique-contribution map (`compute_unique_contribution`,
run separately per pipeline via `run_unique_contribution_both_pipelines`),
compared directly - topomap trio (raw's own map, SI-GEVD's own map,
significance-masked difference), the Wilcoxon test on the ROI-averaged
values, and a per-subject swarm of the SI-GEVD-minus-raw improvement.

In [ ]:
def show_pipeline_comparison_topomap_trio(diff_raw, diff_sigevd, label, epoch, predictor, metric=METRIC):
    "Group-averaged topomaps: raw's own unique-contribution map, SI-GEVD's own map, and their per-sensor-Wilcoxon-significance-masked difference"
    pct = metric == 'ev'
    topo_kwargs = dict(pct=pct, condition=epoch, predictor=predictor, colorbar_side='left', colorbar_label=False, **TOPO_ARGS)
    mean_raw = diff_raw[metric].mean('case')
    mean_sigevd = diff_sigevd[metric].mean('case')
    topomap_with_colorbar(None, mean_raw, label=f'{label}: raw', **topo_kwargs)
    topomap_with_colorbar(None, mean_sigevd, label=f'{label}: SI-GEVD', **topo_kwargs)
    a = diff_raw[metric].get_data(('case', 'sensor'))
    b = diff_sigevd[metric].get_data(('case', 'sensor'))
    significant = per_sensor_wilcoxon_mask(a, b)
    mean_diff = mean_sigevd - mean_raw
    masked_diff = NDVar(numpy.ma.MaskedArray(mean_diff.x, mask=~significant), mean_diff.dims, mean_diff.name, mean_diff.info)
    topomap_with_colorbar(None, masked_diff, label=f'{label}: SI-GEVD - raw', **topo_kwargs)


def show_pipeline_comparison_swarm(diff_raw, diff_sigevd, label, epoch, metric=METRIC):
    "One dot per subject: SI-GEVD minus raw, ROI-averaged (mirrors cortical_analysis_2.ipynb's show_subject_swarm)"
    improvement = diff_sigevd[f'roi_{metric}'].x - diff_raw[f'roi_{metric}'].x
    fig, ax = plt.subplots(figsize=(2, 2.5))
    sns.swarmplot(y=improvement, ax=ax, size=5, color='.25')
    ax.axhline(0, color='k', lw=0.8, ls='--')
    ax.set_ylabel(f'SI-GEVD - raw ({metric})')
    ax.set_xticks([])
    ax.set_title(f'{label} ({epoch})', fontsize=9)
    fig.tight_layout()
    return fig


def section_pipeline_tracking(full_model, predictor, label, epoch, metric='r'):
    diff_raw, diff_sigevd = run_unique_contribution_both_pipelines(full_model, predictor, epoch, metric=metric)
    show_pipeline_comparison_topomap_trio(diff_raw, diff_sigevd, label, epoch, predictor, metric=metric)
    stat, p, sig = paired_wilcoxon(diff_raw[f'roi_{metric}'].x, diff_sigevd[f'roi_{metric}'].x)
    print(f"{label} ({epoch}), {metric}: raw mean={diff_raw[f'roi_{metric}'].mean():.4g}, SI-GEVD mean={diff_sigevd[f'roi_{metric}'].mean():.4g}, Wilcoxon W={stat:.3f}, p={p:.4f}, significant (p<0.025)={sig}")
    show_pipeline_comparison_swarm(diff_raw, diff_sigevd, label, epoch, metric=metric)

### Sub-question 4: attention decoding accuracy (two-talker conditions only)

Leave-one-trial-out, Biesmans et al. 2017-style: fit one TRF (ridge
regression, `sigevd.trf_estim`/`lag_data` - the same machinery
`find_sigevd_filter` itself uses, not eelbrain's `boosting()`) on every
training trial's attended stream pooled together, then for each held-out
trial, cut it into non-overlapping windows at each of several trial
lengths and classify each window as "attended" whichever stream (attended
or ignored) the trained TRF's prediction correlates better with the real
EEG. Not present anywhere in `cortical_analysis_2.ipynb` -
`sigevd_eval.decode_attention_accuracy` is new, validated on synthetic
data (see its own docstring), not against real EEG.

**This is the slowest section in the notebook** - a full leave-one-trial-
out TRF refit per subject, per condition, per predictor, per pipeline.

In [ ]:
TRIAL_LENGTHS = (5, 10, 20, 40, 60)  # seconds - not tuned, a reasonable-looking sweep


def compute_decoding_accuracy_per_subject(pipeline, params, epoch, predictor_name, trial_lengths=TRIAL_LENGTHS):
    "Per-subject leave-one-trial-out decoding accuracy at each trial length, for one condition/predictor/pipeline"
    subjects = pipeline.get_field_values('subject')
    per_subject = {length: [] for length in trial_lengths}
    for subject in subjects:
        trials = load_decoding_trials(pipeline, subject, epoch, predictor_name, params['raw'], params['samplingrate'])
        accuracy = decode_attention_accuracy(trials, params['samplingrate'], LAG_START_MS, LAG_STOP_MS, LAM, trial_lengths)
        for length in trial_lengths:
            per_subject[length].append(accuracy[length])
    return {length: numpy.array(values) for length, values in per_subject.items()}


def section_attention_decoding(predictor_name, label, epoch, trial_lengths=TRIAL_LENGTHS):
    acc_raw = compute_decoding_accuracy_per_subject(e_raw, PARAMETERS_RAW, epoch, predictor_name, trial_lengths)
    acc_sigevd = compute_decoding_accuracy_per_subject(e_sigevd, PARAMETERS_SIGEVD, epoch, predictor_name, trial_lengths)
    for length in trial_lengths:
        stat, p, sig = paired_wilcoxon(acc_raw[length], acc_sigevd[length])
        print(f"{label} decoding ({epoch}), {length}s: raw mean={acc_raw[length].mean() * 100:.1f}%, SI-GEVD mean={acc_sigevd[length].mean() * 100:.1f}%, Wilcoxon W={stat:.3f}, p={p:.4f}, significant (p<0.025)={sig}")

    fig, ax = plt.subplots(figsize=(4, 3))
    ax.plot(trial_lengths, [acc_raw[length].mean() * 100 for length in trial_lengths], marker='o', label='Raw')
    ax.plot(trial_lengths, [acc_sigevd[length].mean() * 100 for length in trial_lengths], marker='o', label='SI-GEVD')
    ax.axhline(50, color='k', lw=0.8, ls='--')
    ax.set_xlabel('Trial length (s)')
    ax.set_ylabel('Decoding accuracy (%)')
    ax.set_title(f'{label} attention decoding ({epoch})', fontsize=9)
    ax.legend()
    fig.tight_layout()
    return fig

## Clean

In [ ]:
data_full_raw = e_raw.load_trfs('all', FULL_CLEAN, epoch='clean', **PARAMETERS_RAW)
data_full_sigevd = e_sigevd.load_trfs('all', FULL_CLEAN, epoch='clean', **PARAMETERS_SIGEVD)

### Sub-question 1: TRF estimation accuracy

In [ ]:
p = section_trf_accuracy(data_full_raw, data_full_sigevd, 'gammatone_on_8', 'Onset', 'clean')

In [ ]:
p = section_trf_accuracy(data_full_raw, data_full_sigevd, 'gammatone_8', 'Envelope', 'clean')

### Sub-question 2: neural tracking (Pearson r)

In [ ]:
section_pipeline_tracking(FULL_CLEAN, onset(ATTEND), 'Onset', 'clean', metric='r')

In [ ]:
section_pipeline_tracking(FULL_CLEAN, envelope(ATTEND), 'Envelope', 'clean', metric='r')

### Sub-question 3: neural tracking (proportion of variance explained)

In [ ]:
section_pipeline_tracking(FULL_CLEAN, onset(ATTEND), 'Onset', 'clean', metric='ev')

In [ ]:
section_pipeline_tracking(FULL_CLEAN, envelope(ATTEND), 'Envelope', 'clean', metric='ev')

## Diotic

In [ ]:
data_full_raw = e_raw.load_trfs('all', FULL_BASE, epoch='diotic', **PARAMETERS_RAW)
data_full_sigevd = e_sigevd.load_trfs('all', FULL_BASE, epoch='diotic', **PARAMETERS_SIGEVD)

### Sub-question 1: TRF estimation accuracy

In [ ]:
p = section_trf_accuracy(data_full_raw, data_full_sigevd, 'gammatone_on_8', 'Onset', 'diotic')

In [ ]:
p = section_trf_accuracy(data_full_raw, data_full_sigevd, 'gammatone_8', 'Envelope', 'diotic')

### Sub-question 2: neural tracking (Pearson r)

In [ ]:
section_pipeline_tracking(FULL_BASE, onset(ATTEND), 'Onset', 'diotic', metric='r')

In [ ]:
section_pipeline_tracking(FULL_BASE, envelope(ATTEND), 'Envelope', 'diotic', metric='r')

### Sub-question 3: neural tracking (proportion of variance explained)

In [ ]:
section_pipeline_tracking(FULL_BASE, onset(ATTEND), 'Onset', 'diotic', metric='ev')

In [ ]:
section_pipeline_tracking(FULL_BASE, envelope(ATTEND), 'Envelope', 'diotic', metric='ev')

### Sub-question 4: attention decoding accuracy

In [ ]:
p = section_attention_decoding('gammatone-on-8', 'Onset', 'diotic')

In [ ]:
p = section_attention_decoding('gammatone-8', 'Envelope', 'diotic')

## Dichotic Left

In [ ]:
data_full_raw = e_raw.load_trfs('all', FULL_BASE, epoch='dichotic-left', **PARAMETERS_RAW)
data_full_sigevd = e_sigevd.load_trfs('all', FULL_BASE, epoch='dichotic-left', **PARAMETERS_SIGEVD)

### Sub-question 1: TRF estimation accuracy

In [ ]:
p = section_trf_accuracy(data_full_raw, data_full_sigevd, 'gammatone_on_8', 'Onset', 'dichotic-left')

In [ ]:
p = section_trf_accuracy(data_full_raw, data_full_sigevd, 'gammatone_8', 'Envelope', 'dichotic-left')

### Sub-question 2: neural tracking (Pearson r)

In [ ]:
section_pipeline_tracking(FULL_BASE, onset(ATTEND), 'Onset', 'dichotic-left', metric='r')

In [ ]:
section_pipeline_tracking(FULL_BASE, envelope(ATTEND), 'Envelope', 'dichotic-left', metric='r')

### Sub-question 3: neural tracking (proportion of variance explained)

In [ ]:
section_pipeline_tracking(FULL_BASE, onset(ATTEND), 'Onset', 'dichotic-left', metric='ev')

In [ ]:
section_pipeline_tracking(FULL_BASE, envelope(ATTEND), 'Envelope', 'dichotic-left', metric='ev')

### Sub-question 4: attention decoding accuracy

In [ ]:
p = section_attention_decoding('gammatone-on-8', 'Onset', 'dichotic-left')

In [ ]:
p = section_attention_decoding('gammatone-8', 'Envelope', 'dichotic-left')

## Dichotic Right

In [ ]:
data_full_raw = e_raw.load_trfs('all', FULL_BASE, epoch='dichotic-right', **PARAMETERS_RAW)
data_full_sigevd = e_sigevd.load_trfs('all', FULL_BASE, epoch='dichotic-right', **PARAMETERS_SIGEVD)

### Sub-question 1: TRF estimation accuracy

In [ ]:
p = section_trf_accuracy(data_full_raw, data_full_sigevd, 'gammatone_on_8', 'Onset', 'dichotic-right')

In [ ]:
p = section_trf_accuracy(data_full_raw, data_full_sigevd, 'gammatone_8', 'Envelope', 'dichotic-right')

### Sub-question 2: neural tracking (Pearson r)

In [ ]:
section_pipeline_tracking(FULL_BASE, onset(ATTEND), 'Onset', 'dichotic-right', metric='r')

In [ ]:
section_pipeline_tracking(FULL_BASE, envelope(ATTEND), 'Envelope', 'dichotic-right', metric='r')

### Sub-question 3: neural tracking (proportion of variance explained)

In [ ]:
section_pipeline_tracking(FULL_BASE, onset(ATTEND), 'Onset', 'dichotic-right', metric='ev')

In [ ]:
section_pipeline_tracking(FULL_BASE, envelope(ATTEND), 'Envelope', 'dichotic-right', metric='ev')

### Sub-question 4: attention decoding accuracy

In [ ]:
p = section_attention_decoding('gammatone-on-8', 'Onset', 'dichotic-right')

In [ ]:
p = section_attention_decoding('gammatone-8', 'Envelope', 'dichotic-right')

## Binaural

In [ ]:
data_full_raw = e_raw.load_trfs('all', FULL_BASE, epoch='binaural', **PARAMETERS_RAW)
data_full_sigevd = e_sigevd.load_trfs('all', FULL_BASE, epoch='binaural', **PARAMETERS_SIGEVD)

### Sub-question 1: TRF estimation accuracy

In [ ]:
p = section_trf_accuracy(data_full_raw, data_full_sigevd, 'gammatone_on_8', 'Onset', 'binaural')

In [ ]:
p = section_trf_accuracy(data_full_raw, data_full_sigevd, 'gammatone_8', 'Envelope', 'binaural')

### Sub-question 2: neural tracking (Pearson r)

In [ ]:
section_pipeline_tracking(FULL_BASE, onset(ATTEND), 'Onset', 'binaural', metric='r')

In [ ]:
section_pipeline_tracking(FULL_BASE, envelope(ATTEND), 'Envelope', 'binaural', metric='r')

### Sub-question 3: neural tracking (proportion of variance explained)

In [ ]:
section_pipeline_tracking(FULL_BASE, onset(ATTEND), 'Onset', 'binaural', metric='ev')

In [ ]:
section_pipeline_tracking(FULL_BASE, envelope(ATTEND), 'Envelope', 'binaural', metric='ev')

### Sub-question 4: attention decoding accuracy

In [ ]:
p = section_attention_decoding('gammatone-on-8', 'Onset', 'binaural')

In [ ]:
p = section_attention_decoding('gammatone-8', 'Envelope', 'binaural')